## Downloading USGS stream flow and water level data

In [54]:
import datetime as dt
import os
import polars as pl
import pandas as pd
from dataretrieval import waterdata # Python wrapper for USGS' Water Data APIs

In [2]:
API_USGS_PAT = os.getenv('API_KEY_USGS')

if API_USGS_PAT:
    os.environ['API_KEY_USGS'] = API_USGS_PAT
else:
    print("No USGS API key detected in .env under API_KEY_USGS.")

In [3]:
dt.datetime.today().date().strftime('%Y-%m-%d')

'2026-07-07'

In [40]:
FPATH_DATA = '../data/'
FPATH_DATA_STATIONS = FPATH_DATA + 'stations.csv'

STATISTIC_ID_MEAN = '00003'
PARAMETER_CODE_DISCHARGE = '00060'
PARAMETER_CODE_GAGEHEIGHT = '00065'
PARAMETER_CODES = [PARAMETER_CODE_DISCHARGE, PARAMETER_CODE_GAGEHEIGHT]

TIME_EARLIEST = '1910-10-01'
TIME_LATEST = ''
TIME_FULL = '2024-03-01/2024-03-05'

### Load monitoring stations of interest

In [21]:
stations_csv = pl.read_csv(FPATH_DATA_STATIONS)

In [ ]:
# 'is_disjoint' denotes monitoring stations of interest for this analysis
station_ids = stations_csv.filter(pl.col('is_disjoint') == True)['station_id'].to_list()
station_ids

['USGS-12370000',
 'USGS-12371550',
 'USGS-12372000',
 'USGS-12365700',
 'USGS-12366000',
 'USGS-12362500',
 'USGS-12362000',
 'USGS-12358500',
 'USGS-12355500']

In [9]:
stations_usgs = waterdata.get_monitoring_locations(
    monitoring_location_id=station_ids,
    
)

Retrieving: monitoring-locations · 1 page · 9 rows · 998/1,000 requests remaining


In [ ]:
# Find which data are measured at each station
time_series_usgs = waterdata.get_time_series_metadata(
    monitoring_location_id=station_ids,
    skip_geometry=True
)[0]

Retrieving: time-series-metadata · 1 page · 64 rows · 997/1,000 requests remaining


In [44]:
time_series_usgs.columns

Index(['time_series_id', 'unit_of_measure', 'parameter_name', 'parameter_code',
       'statistic_id', 'hydrologic_unit_code', 'state_name', 'last_modified',
       'begin', 'end', 'begin_utc', 'end_utc', 'computation_period_identifier',
       'computation_identifier', 'thresholds', 'sublocation_identifier',
       'primary', 'monitoring_location_id', 'web_description',
       'parameter_description', 'parent_time_series_id', 'data_gap_interval'],
      dtype='str')

In [90]:
# What are all of the data types observed at our stations? Water temp, discharge, etc.
time_series_usgs.loc[
    :, 
    ['parameter_name', 'parameter_code', 'parameter_description', 'unit_of_measure', 'statistic_id']
].drop_duplicates() # monitoring_location_id
# Gage height: Water level relative to local reference point
# Elevation, lake/res, NGVD29: Water level relative to an absolute reference point
# Reservoir height: Also an absolute measure

,parameter_name,parameter_code,parameter_description,unit_of_measure,statistic_id
0,Gage height,00065,"Gage height, feet",ft,00011
1,Discharge,00060,"Discharge, cubic feet per second",ft^3/s,NaN
2,Discharge,00060,"Discharge, cubic feet per second",ft^3/s,00011
3,"Temperature, water",00010,"Temperature, water, degrees Celsius",degC,00001
5,"Temperature, water",00010,"Temperature, water, degrees Celsius",degC,00011
6,"Elevation, lake/res, NGVD29",62614,Lake or reservoir water surface elevation abov...,ft,00011
9,Specific cond at 25C,00095,"Specific conductance, water, unfiltered, micro...",uS/cm,00011
11,"Elevation, lake/res, NGVD29",62614,Lake or reservoir water surface elevation abov...,ft,32359
12,"Temperature, water",00010,"Temperature, water, degrees Celsius",degC,00003
13,Suspnd sedmnt disch,80155,"Suspended sediment discharge, short tons per day",tons/day,00003


In [94]:
# Find which values (discharge, gage height) and for which dates we have MEAN data at each station
time_series_available = time_series_usgs.loc[
    time_series_usgs['parameter_code'].isin(PARAMETER_CODES) & (time_series_usgs['statistic_id'] == STATISTIC_ID_MEAN), 
    [
        'monitoring_location_id', 
        'time_series_id', 
        'parameter_name', 
        'parameter_code', 
        'statistic_id', 
        'unit_of_measure', 
        'begin', 
        'end'
    ]
]

In [70]:
stations_csv_pd = pd.read_csv(FPATH_DATA_STATIONS)
# stations_csv_pd.rename(columns={'station_id':'monitoring_location_id'}, inplace=True)
# stations_csv.select(['station_name', 'station_shorthand', 'station_id'])

In [95]:
time_series_available = time_series_available.merge(
    stations_csv_pd.loc[:, ['station_name', 'station_shorthand', 'station_id']],
    left_on = 'monitoring_location_id',
    right_on = 'station_id'
)

In [96]:
time_series_available.sort_values(by = ['monitoring_location_id'])

,monitoring_location_id,time_series_id,parameter_name,parameter_code,statistic_id,unit_of_measure,begin,end,station_name,station_shorthand,station_id
1,USGS-12355500,7044c60248554783823aabb670451c24,Discharge,00060,00003,ft^3/s,1910-10-01 00:00:00.000001,2026-07-05 00:00:00.000001,N F Flathead River nr Columbia Falls MT,fhr_n,USGS-12355500
0,USGS-12358500,587e53501ee04585bd078dfafed6b54b,Discharge,00060,00003,ft^3/s,1939-10-01 00:00:00.000001,2026-07-05 00:00:00.000001,M F Flathead River near West Glacier MT,fhr_m,USGS-12358500
4,USGS-12362500,807167eeb3ec416e85b4cd277a41670a,Discharge,00060,00003,ft^3/s,1911-02-01 00:00:00.000001,2026-07-05 00:00:00.000001,S F Flathead River nr Columbia Falls MT,fhr_s,USGS-12362500
5,USGS-12365700,9b9c37eb614d4a6d9fcb938befc5ca07,Discharge,00060,00003,ft^3/s,2007-03-01 00:00:00.000001,2026-07-05 00:00:00.000001,"Stillwater River at Lawrence Park, at Kalispell",stillwater,USGS-12365700
6,USGS-12366000,bbeb6445470d427f84aa6ec49e7c66b1,Discharge,00060,00003,ft^3/s,1928-08-01 00:00:00.000001,2026-07-05 00:00:00.000001,Whitefish River near Kalispell MT,whitefish,USGS-12366000
2,USGS-12370000,7b04109277f74a648002d475466fec30,Discharge,00060,00003,ft^3/s,1922-05-01 00:00:00.000001,2026-07-05 00:00:00.000001,"Swan River near Bigfork, MT",swan,USGS-12370000
3,USGS-12372000,7db63fb262c845f7904cd34285cc4c21,Discharge,00060,00003,ft^3/s,1907-08-01 00:00:00.000001,2026-07-05 00:00:00.000001,Flathead River near Polson MT,fhr_out,USGS-12372000


### Water time series data

In [ ]:
discharge = waterdata.get_daily(
    monitoring_location_id=station_ids,
    parameter_code=PARAMETER_CODE_DISCHARGE,
    statistic_id=STATISTIC_ID_MEAN,
    skip_geometry=True
)

station_id
str
"""USGS-12370000"""
"""USGS-12371550"""
"""USGS-12372000"""
"""USGS-12365700"""
"""USGS-12366000"""
"""USGS-12362500"""
"""USGS-12362000"""
"""USGS-12358500"""
"""USGS-12355500"""
